In [ ]:
# ============================================================
# PMSM THERMAL MODEL IDENTIFICATION
# STEP 2
#
# Goal:
# Determine whether the proposed four-node thermal model
# is physically and statistically defensible using the
# 1.3M-record dataset.
#
# IMPORTANT:
# This code does NOT train a neural network.
# It identifies the thermal equations first.
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression

# ============================================================
# CONFIGURATION
# ============================================================

DATA_PATH = (
    "/Users/prashantsingh.basnet@iqvia.com/Documents/sim/.venv/measures_v2.csv"
    "electric-motor-temperature/measures_v2.csv"
)

DT = 0.5                     # seconds; dataset sampling = 2 Hz

R_S0 = 0.05                  # provisional only
T0 = 20.0                    # reference temperature
ALPHA = 0.00393              # copper temperature coefficient

OUTPUT_DIR = "/kaggle/working/thermal_identification"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# ============================================================
# COLUMN DEFINITIONS
# ============================================================

REQUIRED_COLUMNS = [
    "u_q",
    "coolant",
    "stator_winding",
    "u_d",
    "stator_tooth",
    "motor_speed",
    "i_d",
    "i_q",
    "pm",
    "stator_yoke",
    "ambient",
    "torque",
    "profile_id"
]

TEMPERATURE_COLUMNS = [
    "stator_winding",
    "stator_tooth",
    "stator_yoke",
    "pm"
]

# ============================================================
# LOAD DATA
# ============================================================

print("=" * 80)
print("PMSM THERMAL MODEL IDENTIFICATION")
print("=" * 80)

print("\nLoading dataset...")

df = pd.read_csv(DATA_PATH)

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")

missing = [
    c for c in REQUIRED_COLUMNS
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

df = df[REQUIRED_COLUMNS].copy()

# ============================================================
# CLEAN DATA
# ============================================================

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

before = len(df)

df = df.dropna().copy()

after = len(df)

print(
    f"\nRemoved invalid rows: "
    f"{before - after:,}"
)

# ============================================================
# SORT BY PROFILE
# ============================================================

df = df.sort_values(
    ["profile_id"]
).reset_index(drop=True)

print(
    f"Profiles: "
    f"{df['profile_id'].nunique()}"
)

# ============================================================
# TEMPERATURE-DEPENDENT RESISTANCE
# ============================================================

df["R_s"] = (
    R_S0
    *
    (
        1.0
        +
        ALPHA
        *
        (
            df["stator_winding"] - T0
        )
    )
)

# ============================================================
# COPPER LOSS
#
# Assuming i_d/i_q are peak dq currents:
#
# P_cu = 3/2 * R_s * (id^2 + iq^2)
# ============================================================

df["P_copper"] = (
    1.5
    *
    df["R_s"]
    *
    (
        df["i_d"] ** 2
        +
        df["i_q"] ** 2
    )
)

# ============================================================
# CENTRAL DIFFERENCE FUNCTION
# ============================================================

def add_profile_derivative(
    dataframe,
    column,
    dt
):
    """
    Calculate temporal derivative independently
    inside each profile.

    Central difference:

        dT/dt ≈ (T[k+1] - T[k-1]) / (2dt)

    Forward/backward differences are used at
    profile boundaries.
    """

    result = np.full(
        len(dataframe),
        np.nan,
        dtype=np.float64
    )

    for profile_id, group in dataframe.groupby(
        "profile_id",
        sort=False
    ):

        idx = group.index.to_numpy()

        values = group[column].to_numpy(
            dtype=np.float64
        )

        n = len(values)

        if n < 2:
            continue

        # Forward difference
        result[idx[0]] = (
            values[1] - values[0]
        ) / dt

        # Central difference
        if n > 2:

            result[idx[1:-1]] = (
                values[2:] - values[:-2]
            ) / (2.0 * dt)

        # Backward difference
        result[idx[-1]] = (
            values[-1] - values[-2]
        ) / dt

    return result


# ============================================================
# COMPUTE TEMPERATURE DERIVATIVES
# ============================================================

print("\nCalculating temperature derivatives...")

for column in TEMPERATURE_COLUMNS:

    derivative_column = (
        "d" + column + "_dt"
    )

    df[derivative_column] = (
        add_profile_derivative(
            df,
            column,
            DT
        )
    )

# ============================================================
# DISPLAY DERIVATIVE STATISTICS
# ============================================================

print("\n" + "=" * 80)
print("TEMPERATURE DERIVATIVE STATISTICS")
print("=" * 80)

derivative_columns = [
    "dstator_winding_dt",
    "dstator_tooth_dt",
    "dstator_yoke_dt",
    "dpm_dt"
]

print(
    df[derivative_columns]
    .describe()
    .T
)

# ============================================================
# REMOVE DERIVATIVE INVALID ROWS
# ============================================================

analysis_columns = (
    [
        "P_copper",
        "stator_winding",
        "stator_tooth",
        "stator_yoke",
        "pm",
        "coolant",
        "ambient"
    ]
    +
    derivative_columns
)

analysis_df = df.dropna(
    subset=analysis_columns
).copy()

print(
    f"\nRows available for identification: "
    f"{len(analysis_df):,}"
)

# ============================================================
# ============================================================
# MODEL 1
#
# SIMPLE WINDING THERMAL MODEL
#
# Cw * dTw/dt =
#
#       P_copper
#       -
#       Gwt * (Tw - Tt)
#
# Rearrange:
#
# dTw/dt =
#
#       (1/Cw) * P_copper
#       -
#       (Gwt/Cw) * (Tw-Tt)
#
# Let:
#
# a1 = 1/Cw
# a2 = Gwt/Cw
#
# Then:
#
# dTw/dt = a1*Pcu - a2*(Tw-Tt)
#
# ============================================================
# ============================================================

print("\n" + "=" * 80)
print("MODEL 1: WINDING THERMAL DYNAMICS")
print("=" * 80)

X_winding = np.column_stack(
    [
        analysis_df["P_copper"].to_numpy(),
        (
            analysis_df["stator_winding"]
            -
            analysis_df["stator_tooth"]
        ).to_numpy()
    ]
)

y_winding = (
    analysis_df[
        "dstator_winding_dt"
    ].to_numpy()
)

model_winding = LinearRegression(
    fit_intercept=False
)

model_winding.fit(
    X_winding,
    y_winding
)

a1 = model_winding.coef_[0]
a2 = model_winding.coef_[1]

print("\nEstimated coefficients:")

print(
    f"1/Cw       = {a1:.12e}"
)

print(
    f"Gwt/Cw     = {a2:.12e}"
)

# ============================================================
# RECOVER PHYSICAL PARAMETERS
# ============================================================

if abs(a1) > 1e-15:

    Cw = 1.0 / a1

else:

    Cw = np.nan

if abs(a1) > 1e-15:

    Gwt = a2 / a1

else:

    Gwt = np.nan

print("\nRecovered physical parameters:")

print(
    f"Cw  = {Cw:.12e}"
)

print(
    f"Gwt = {Gwt:.12e}"
)

# ============================================================
# MODEL PERFORMANCE
# ============================================================

pred_winding = model_winding.predict(
    X_winding
)

rmse_winding = np.sqrt(
    mean_squared_error(
        y_winding,
        pred_winding
    )
)

mae_winding = mean_absolute_error(
    y_winding,
    pred_winding
)

r2_winding = r2_score(
    y_winding,
    pred_winding
)

print("\nWinding derivative reconstruction:")

print(
    f"RMSE = {rmse_winding:.8f} °C/s"
)

print(
    f"MAE  = {mae_winding:.8f} °C/s"
)

print(
    f"R²   = {r2_winding:.8f}"
)

# ============================================================
# ============================================================
# MODEL 2
#
# WINDING + TOOTH
#
# Cw*dTw/dt =
# Pcu - Gwt(Tw-Tt)
#
# Ct*dTt/dt =
# Gwt(Tw-Tt)
# - Gty(Tt-Ty)
# - Gtp(Tt-Tpm)
#
# ============================================================
# ============================================================

print("\n" + "=" * 80)
print("MODEL 2: WINDING + TOOTH")
print("=" * 80)

# ------------------------------------------------------------
# Winding equation
# ------------------------------------------------------------

X_w = np.column_stack(
    [
        analysis_df["P_copper"].to_numpy(),

        (
            analysis_df["stator_winding"]
            -
            analysis_df["stator_tooth"]
        ).to_numpy()
    ]
)

y_w = analysis_df[
    "dstator_winding_dt"
].to_numpy()

# ------------------------------------------------------------
# Tooth equation
#
# dTt/dt =
#
# a*(Tw-Tt)
# - b*(Tt-Ty)
# - c*(Tt-Tpm)
#
# ------------------------------------------------------------

X_t = np.column_stack(
    [
        (
            analysis_df["stator_winding"]
            -
            analysis_df["stator_tooth"]
        ).to_numpy(),

        (
            analysis_df["stator_tooth"]
            -
            analysis_df["stator_yoke"]
        ).to_numpy(),

        (
            analysis_df["stator_tooth"]
            -
            analysis_df["pm"]
        ).to_numpy()
    ]
)

y_t = analysis_df[
    "dstator_tooth_dt"
].to_numpy()

model_w = LinearRegression(
    fit_intercept=False
)

model_t = LinearRegression(
    fit_intercept=False
)

model_w.fit(
    X_w,
    y_w
)

model_t.fit(
    X_t,
    y_t
)

# ============================================================
# COEFFICIENTS
# ============================================================

a_w = model_w.coef_[0]
b_w = model_w.coef_[1]

a_t = model_t.coef_[0]
b_t = model_t.coef_[1]
c_t = model_t.coef_[2]

print("\nWinding coefficients:")
print(
    f"Pcu coefficient      = {a_w:.12e}"
)
print(
    f"Temperature coupling = {b_w:.12e}"
)

print("\nTooth coefficients:")
print(
    f"Tw-Tt coefficient    = {a_t:.12e}"
)
print(
    f"Tt-Ty coefficient    = {b_t:.12e}"
)
print(
    f"Tt-Tpm coefficient   = {c_t:.12e}"
)

# ============================================================
# PERFORMANCE
# ============================================================

pred_w = model_w.predict(X_w)
pred_t = model_t.predict(X_t)

metrics_model2 = pd.DataFrame(
    {
        "state": [
            "stator_winding",
            "stator_tooth"
        ],

        "RMSE_C_per_s": [
            np.sqrt(
                mean_squared_error(
                    y_w,
                    pred_w
                )
            ),

            np.sqrt(
                mean_squared_error(
                    y_t,
                    pred_t
                )
            )
        ],

        "MAE_C_per_s": [
            mean_absolute_error(
                y_w,
                pred_w
            ),

            mean_absolute_error(
                y_t,
                pred_t
            )
        ],

        "R2": [
            r2_score(
                y_w,
                pred_w
            ),

            r2_score(
                y_t,
                pred_t
            )
        ]
    }
)

print("\nModel 2 performance:")
print(metrics_model2)

# ============================================================
# ============================================================
# MODEL 3
#
# FULL FOUR-NODE MODEL
#
# ------------------------------------------------------------
#
# Cw*dTw/dt =
#
# Pcu
# - Gwt(Tw-Tt)
#
# ------------------------------------------------------------
#
# Ct*dTt/dt =
#
# Gwt(Tw-Tt)
# - Gty(Tt-Ty)
# - Gtp(Tt-Tpm)
#
# ------------------------------------------------------------
#
# Cy*dTy/dt =
#
# Gty(Tt-Ty)
# - Gyc(Ty-Tc)
# - Gya(Ty-Ta)
#
# ------------------------------------------------------------
#
# Cpm*dTpm/dt =
#
# Gtp(Tt-Tpm)
# - Gpa(Tpm-Ta)
#
# ============================================================
# ============================================================

print("\n" + "=" * 80)
print("MODEL 3: FULL FOUR-NODE THERMAL NETWORK")
print("=" * 80)

# ============================================================
# WINDING
# ============================================================

X_w = np.column_stack(
    [
        analysis_df["P_copper"].to_numpy(),

        (
            analysis_df["stator_winding"]
            -
            analysis_df["stator_tooth"]
        ).to_numpy()
    ]
)

y_w = analysis_df[
    "dstator_winding_dt"
].to_numpy()

# ============================================================
# TOOTH
# ============================================================

X_t = np.column_stack(
    [
        (
            analysis_df["stator_winding"]
            -
            analysis_df["stator_tooth"]
        ).to_numpy(),

        (
            analysis_df["stator_tooth"]
            -
            analysis_df["stator_yoke"]
        ).to_numpy(),

        (
            analysis_df["stator_tooth"]
            -
            analysis_df["pm"]
        ).to_numpy()
    ]
)

y_t = analysis_df[
    "dstator_tooth_dt"
].to_numpy()

# ============================================================
# YOKE
# ============================================================

X_y = np.column_stack(
    [
        (
            analysis_df["stator_tooth"]
            -
            analysis_df["stator_yoke"]
        ).to_numpy(),

        (
            analysis_df["stator_yoke"]
            -
            analysis_df["coolant"]
        ).to_numpy(),

        (
            analysis_df["stator_yoke"]
            -
            analysis_df["ambient"]
        ).to_numpy()
    ]
)

y_y = analysis_df[
    "dstator_yoke_dt"
].to_numpy()

# ============================================================
# PM
# ============================================================

X_pm = np.column_stack(
    [
        (
            analysis_df["stator_tooth"]
            -
            analysis_df["pm"]
        ).to_numpy(),

        (
            analysis_df["pm"]
            -
            analysis_df["ambient"]
        ).to_numpy()
    ]
)

y_pm = analysis_df[
    "dpm_dt"
].to_numpy()

# ============================================================
# FIT
# ============================================================

model_w = LinearRegression(
    fit_intercept=False
)

model_t = LinearRegression(
    fit_intercept=False
)

model_y = LinearRegression(
    fit_intercept=False
)

model_pm = LinearRegression(
    fit_intercept=False
)

model_w.fit(X_w, y_w)
model_t.fit(X_t, y_t)
model_y.fit(X_y, y_y)
model_pm.fit(X_pm, y_pm)

# ============================================================
# COEFFICIENTS
# ============================================================

print("\nEstimated dynamic coefficients")
print("-" * 60)

print("\nWINDING")

print(
    f"Pcu -> dTw/dt      : "
    f"{model_w.coef_[0]:.12e}"
)

print(
    f"Tw-Tt -> dTw/dt    : "
    f"{model_w.coef_[1]:.12e}"
)

print("\nTOOTH")

print(
    f"Tw-Tt -> dTt/dt    : "
    f"{model_t.coef_[0]:.12e}"
)

print(
    f"Tt-Ty -> dTt/dt    : "
    f"{model_t.coef_[1]:.12e}"
)

print(
    f"Tt-Tpm -> dTt/dt   : "
    f"{model_t.coef_[2]:.12e}"
)

print("\nYOKE")

print(
    f"Tt-Ty -> dTy/dt    : "
    f"{model_y.coef_[0]:.12e}"
)

print(
    f"Ty-Tc -> dTy/dt    : "
    f"{model_y.coef_[1]:.12e}"
)

print(
    f"Ty-Ta -> dTy/dt    : "
    f"{model_y.coef_[2]:.12e}"
)

print("\nPM")

print(
    f"Tt-Tpm -> dTpm/dt  : "
    f"{model_pm.coef_[0]:.12e}"
)

print(
    f"Tpm-Ta -> dTpm/dt  : "
    f"{model_pm.coef_[1]:.12e}"
)

# ============================================================
# PREDICTIONS
# ============================================================

pred_w = model_w.predict(X_w)
pred_t = model_t.predict(X_t)
pred_y = model_y.predict(X_y)
pred_pm = model_pm.predict(X_pm)

# ============================================================
# METRICS
# ============================================================

results = []

for name, actual, predicted in [

    (
        "stator_winding",
        y_w,
        pred_w
    ),

    (
        "stator_tooth",
        y_t,
        pred_t
    ),

    (
        "stator_yoke",
        y_y,
        pred_y
    ),

    (
        "pm",
        y_pm,
        pred_pm
    )
]:

    results.append(
        {
            "state": name,

            "RMSE_C_per_s":
                np.sqrt(
                    mean_squared_error(
                        actual,
                        predicted
                    )
                ),

            "MAE_C_per_s":
                mean_absolute_error(
                    actual,
                    predicted
                ),

            "R2":
                r2_score(
                    actual,
                    predicted
                )
        }
    )

results_df = pd.DataFrame(
    results
)

print("\n" + "=" * 80)
print("FULL THERMAL MODEL PERFORMANCE")
print("=" * 80)

print(
    results_df.to_string(
        index=False
    )
)

# ============================================================
# PHYSICAL SIGN CHECK
# ============================================================

print("\n" + "=" * 80)
print("PHYSICAL SIGN CHECK")
print("=" * 80)

print(
    """
For the proposed passive thermal network, conductances
should normally be non-negative.

Expected qualitative signs:

Winding:
    Pcu coefficient              > 0
    -(Tw-Tt) coefficient         < 0

Tooth:
    +(Tw-Tt) coefficient         > 0
    -(Tt-Ty) coefficient         < 0
    -(Tt-Tpm) coefficient        < 0

Yoke:
    +(Tt-Ty) coefficient         > 0
    -(Ty-Tc) coefficient         < 0
    -(Ty-Ta) coefficient         < 0

PM:
    +(Tt-Tpm) coefficient        > 0
    -(Tpm-Ta) coefficient        < 0
"""
)

print("\nActual coefficients:")

for name, model in [
    ("Winding", model_w),
    ("Tooth", model_t),
    ("Yoke", model_y),
    ("PM", model_pm)
]:

    print(
        f"\n{name}:"
    )

    for i, coef in enumerate(
        model.coef_
    ):

        print(
            f"  coefficient[{i}] = "
            f"{coef:.12e}"
        )

# ============================================================
# CONDITION NUMBERS
# ============================================================

print("\n" + "=" * 80)
print("IDENTIFIABILITY / CONDITION NUMBERS")
print("=" * 80)

design_matrices = {
    "winding": X_w,
    "tooth": X_t,
    "yoke": X_y,
    "pm": X_pm
}

condition_results = []

for name, X in design_matrices.items():

    rank = np.linalg.matrix_rank(X)

    condition = np.linalg.cond(X)

    condition_results.append(
        {
            "equation": name,
            "rows": X.shape[0],
            "features": X.shape[1],
            "rank": rank,
            "condition_number": condition
        }
    )

condition_df = pd.DataFrame(
    condition_results
)

print(
    condition_df.to_string(
        index=False
    )
)

# ============================================================
# FEATURE CORRELATIONS
# ============================================================

print("\n" + "=" * 80)
print("REGRESSION FEATURE CORRELATIONS")
print("=" * 80)

for name, X in design_matrices.items():

    print(
        f"\n{name.upper()}"
    )

    print(
        np.corrcoef(
            X,
            rowvar=False
        )
    )

# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "thermal_model_metrics.csv"
    ),
    index=False
)

condition_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "thermal_identifiability.csv"
    ),
    index=False
)

# ============================================================
# SAVE PARAMETERS
# ============================================================

parameter_records = []

models = {
    "winding": model_w,
    "tooth": model_t,
    "yoke": model_y,
    "pm": model_pm
}

for equation, model in models.items():

    for i, coefficient in enumerate(
        model.coef_
    ):

        parameter_records.append(
            {
                "equation": equation,
                "coefficient_index": i,
                "coefficient": coefficient
            }
        )

parameter_df = pd.DataFrame(
    parameter_records
)

parameter_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "thermal_coefficients.csv"
    ),
    index=False
)

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("IDENTIFICATION COMPLETE")
print("=" * 80)

print(
    f"""
Sampling interval:
    Δt = {DT} seconds

Dataset:
    {len(df):,} records

Profiles:
    {df['profile_id'].nunique()}

Analysis records:
    {len(analysis_df):,}

Results:
    {OUTPUT_DIR}

------------------------------------------------------------

DO NOT IMPLEMENT THE PHYSICS LOSS YET.

The next decision depends on:

1. R² of each thermal equation
2. RMSE of each equation
3. Signs of the estimated coefficients
4. Condition numbers
5. Feature correlations
6. Whether the four-node model is identifiable

Paste the COMPLETE output from this cell.
"""
)